# 미끄러운 미로: 사전학습 → SFT → GRPO

환경의 이동 규칙, 목표로 가는 행동, 위험과 이동 비용을 고려한 행동을 순서대로 학습합니다.
Colab에서 T4 GPU를 선택하고 준비 셀부터 실행하세요.
각 단계는 학습 셀과 바로 아래 평가 셀을 실행해 모델을 직접 확인한 뒤 다음 단계로 넘어갑니다.

- 사전학습: 행동을 지정하고 다음 위치의 예측 확률과 실제 이동을 비교합니다.
- SFT: 미끄러짐을 끄고 켜서 목표로 이동하는 행동을 확인합니다.
- GRPO: 같은 지도에서 SFT와 GRPO 모델을 나란히 실행합니다.

[환경 규칙과 단계별 목표](https://github.com/HisameOgasahara/tmp_full_Training/blob/main/docs/학습설계.md)

## 0. 최신 코드 준비

Colab 런타임을 T4 GPU로 설정한 뒤 실행하세요. 이 셀은 최신 `main`을 새 폴더에 복제합니다.
준비 완료 후 표시되는 커밋을 실험 기록에 저장하세요. 한 실험에서는 처음 가져온 코드를 계속 사용합니다.
Colab에 설치된 PyTorch를 사용합니다.

In [ ]:
import importlib
import io
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request

REPO_URL = "https://github.com/HisameOgasahara/tmp_full_Training.git"
REPO_ROOT = Path(tempfile.mkdtemp(prefix="slippery_maze_")) / "repository"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_ROOT)], check=True)
REVISION = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()

uv_executable = shutil.which("uv")
if uv_executable is None:
    if not sys.platform.startswith("linux"):
        raise RuntimeError("로컬에서는 uv를 설치한 뒤 이 셀을 실행하세요. Colab에서는 자동 준비됩니다.")
    archive_url = "https://github.com/astral-sh/uv/releases/download/0.12.23/uv-x86_64-unknown-linux-gnu.tar.gz"
    data = urllib.request.urlopen(archive_url, timeout=120).read()
    with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as archive:
        member = next(item for item in archive.getmembers() if item.name.endswith("/uv") and item.isfile())
        uv_path = REPO_ROOT.parent / "uv"
        uv_path.write_bytes(archive.extractfile(member).read())
    uv_path.chmod(0o755)
    uv_executable = str(uv_path)

import torch
subprocess.run([uv_executable, "pip", "install", "--system", "--python", sys.executable,
                "numpy>=1.26,<3", "matplotlib>=3.8,<4", "ipywidgets>=8.1,<9"], check=True)
subprocess.run([uv_executable, "pip", "install", "--system", "--python", sys.executable,
                "--no-deps", "--editable", str(REPO_ROOT)], check=True)
for name in list(sys.modules):
    if name == "maze_training" or name.startswith("maze_training."):
        del sys.modules[name]
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
importlib.invalidate_caches()
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
print("실제 실행 커밋:", REVISION)
print("코드 위치:", REPO_ROOT)
print("PyTorch:", torch.__version__)
print("장치:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (t4 프로필은 GPU 권장)")

### 설정

전체 학습은 `PROFILE = "t4"`, 짧은 셀 연결 확인은 `PROFILE = "smoke"`로 실행하세요.
중단한 단계를 이어가려면 같은 설정과 출력 폴더에서 `RESUME = True`를 사용합니다.
새 단계로 넘어갈 때는 `False`로 설정하세요. 같은 단계의 새 학습은 해당 체크포인트와 로그를 덮어씁니다.

In [ ]:
PROFILE = "t4"  # "smoke"로 바꾸면 연결 확인용 짧은 실행
RESUME = False
CONFIG_PATH = str(REPO_ROOT / "configs" / f"{PROFILE}.json")
OUTPUT_ROOT = Path("/content/maze_runs") if Path("/content").exists() else REPO_ROOT / "runs"
OUTPUT_DIR = OUTPUT_ROOT / PROFILE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from maze_training.training import run_stage
from maze_training.notebook import report_world, report_navigation, show_training_curve, launch_world, launch_navigation
from maze_training.runtime import read_config, seed_runtime, create_model, choose_device, save_checkpoint

config = read_config(CONFIG_PATH)
model_for_size, _ = create_model(config, choose_device())
print("파라미터:", f"{sum(p.numel() for p in model_for_size.parameters()):,}")
print("설정:", CONFIG_PATH)
print("체크포인트:", OUTPUT_DIR)
del model_for_size

### 선택: Google Drive에 저장

세션 종료 후 모델을 다시 사용하려면 학습 전에 `USE_DRIVE = True`로 설정하고 실행하세요.
Drive를 연결하면 `/content/drive/MyDrive/maze_training/프로필명`에 저장합니다.

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/maze_training") / PROFILE
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("저장 위치:", OUTPUT_DIR)

## 1. 사전학습 — 행동의 결과를 예측

지도·현재 위치·행동으로 다음 위치의 확률 분포를 예측합니다.
무작위 이동의 전이 기록으로 벽 충돌과 미끄러짐의 규칙을 학습합니다.

In [ ]:
PRETRAIN_PATH = run_stage("pretrain", CONFIG_PATH, OUTPUT_DIR, resume=RESUME)
show_training_curve(OUTPUT_DIR, "pretrain")

### 1단계 평가와 직접 실행

평가 셀을 실행한 뒤 화면에서 행동을 선택하고 한 걸음씩 이동하세요.
모델의 예측 분포와 환경의 정확한 전이확률, 실제 다음 위치를 비교합니다.
TV distance와 KL은 낮을수록, 실제 가능한 위치에 부여한 확률은 높을수록 좋습니다.
확인을 마치면 2단계 학습 셀을 실행하세요.

In [ ]:
PRETRAIN_PATH = str(OUTPUT_DIR / "pretrain.pt")
world_report = report_world(PRETRAIN_PATH)
world_demo = launch_world(PRETRAIN_PATH)

## 2. SFT — 목표로 이동하는 행동을 모방

사전학습 체크포인트를 불러와 BFS가 선택한 다음 행동을 학습합니다.
BFS는 벽·구멍을 피하는 최단 경로를 구하고 미끄러짐은 끈 조건으로 계획합니다.
목표에 도달할 수 있는 여러 현재 위치에서 행동을 학습합니다.

In [ ]:
SFT_PATH = run_stage("sft", CONFIG_PATH, OUTPUT_DIR, resume=RESUME)
show_training_curve(OUTPUT_DIR, "sft")

### 2단계 평가와 직접 실행

평가 셀을 실행해 미끄러짐을 끈 조건과 켠 조건의 도착률·추락률·평균 보상을 비교하세요.
실행 화면에서 한 걸음씩 이동하거나 자동 실행을 사용합니다.
‘미끄러짐 켜기’를 바꾼 뒤 ‘지도 초기화’를 눌러 같은 지도에서 다시 실행하세요.

In [ ]:
SFT_PATH = str(OUTPUT_DIR / "sft.pt")
sft_dry_report = report_navigation({"SFT": SFT_PATH}, CONFIG_PATH, slippery=False)
sft_report = report_navigation({"SFT": SFT_PATH}, CONFIG_PATH, slippery=True)
sft_demo = launch_navigation({"SFT": SFT_PATH}, CONFIG_PATH)

### 2단계 추가 학습 — 기본 길찾기 안정시키기

위 평가에서 미끄러짐을 끈 도착률과 충돌 수를 확인하세요.
아래 셀은 기존 SFT를 총 4,000회까지 이어 학습하고 두 조건을 다시 평가합니다.
더 이어가려면 `SFT_TOTAL_STEPS`를 6,000처럼 늘리고 같은 셀을 실행하세요.
기존 모델이 있는 세션에서는 준비·설정 셀과 저장 위치를 맞춘 뒤 이 셀부터 실행할 수 있습니다.

In [ ]:
SFT_TOTAL_STEPS = 4000 if PROFILE == "t4" else 8
SFT_PATH = run_stage("sft", CONFIG_PATH, OUTPUT_DIR, steps=SFT_TOTAL_STEPS, resume=True)
show_training_curve(OUTPUT_DIR, "sft")
sft_dry_report = report_navigation({"SFT": SFT_PATH}, CONFIG_PATH, slippery=False)
sft_report = report_navigation({"SFT": SFT_PATH}, CONFIG_PATH, slippery=True)
sft_demo = launch_navigation({"SFT": SFT_PATH}, CONFIG_PATH)

## 3. GRPO — 개선된 SFT에서 짧게 학습하고 비교

아래 셀을 실행하면 현재 `sft.pt`에서 GRPO를 새로 시작하고 첫 50회 뒤 검증합니다.
새 실험은 `grpo_refinement` 폴더에 저장합니다. 이 시작 셀을 다시 실행하면 해당 실험을 다시 시작합니다.
이어 학습하려면 다음의 **50회 더 학습** 셀을 사용하세요.

검증 평균 보상이 높은 모델을 `selected_policy.pt`로 보관하고 동점이면 도착률로 선택합니다.
시작 SFT도 선택 후보에 포함합니다. 각 구간의 GRPO 모델과 검증 결과도 따로 저장합니다.
`useful_group_fraction`은 보상 차이가 있어 학습에 사용한 그룹의 비율입니다.

In [ ]:
import json
from maze_training.runtime import write_json

SFT_PATH = str(OUTPUT_DIR / "sft.pt")
GRPO_DIR = OUTPUT_DIR / "grpo_refinement"
GRPO_DIR.mkdir(parents=True, exist_ok=True)
GRPO_INTERVAL = 50 if PROFILE == "t4" else 1
GRPO_LIMIT = config["grpo"]["steps"]
SELECTION_PATH = GRPO_DIR / "selection.json"
SELECTED_PATH = GRPO_DIR / "selected_policy.pt"
baseline_report = report_navigation({"SFT": SFT_PATH}, CONFIG_PATH)
baseline_metrics = baseline_report["policies"]["SFT"]
shutil.copy2(SFT_PATH, GRPO_DIR / "initial_sft.pt")
shutil.copy2(SFT_PATH, SELECTED_PATH)
write_json(SELECTION_PATH, {
    "source": "SFT", "step": 0,
    "metrics": baseline_metrics,
})

def evaluate_and_save_grpo():
    candidate_path = GRPO_DIR / "grpo.pt"
    payload = torch.load(candidate_path, map_location="cpu", weights_only=True)
    step = payload["step"]
    del payload
    report = report_navigation({
        "시작 SFT": str(GRPO_DIR / "initial_sft.pt"),
        "GRPO": str(candidate_path),
    }, CONFIG_PATH)
    metrics = report["policies"]["GRPO"]
    shutil.copy2(candidate_path, GRPO_DIR / f"grpo_{step:04d}.pt")
    write_json(GRPO_DIR / f"validation_{step:04d}.json", report)
    selected = json.loads(SELECTION_PATH.read_text(encoding="utf-8"))
    score = lambda values: (values["mean_return"], values["success_rate"])
    if score(metrics) > score(selected["metrics"]):
        shutil.copy2(candidate_path, SELECTED_PATH)
        selected = {"source": "GRPO", "step": step, "metrics": metrics}
        write_json(SELECTION_PATH, selected)
    print(f"검증 선택: {selected['source']} {selected['step']}회 | "
          f"평균 보상 {selected['metrics']['mean_return']:.3f} | "
          f"도착률 {selected['metrics']['success_rate']:.1%}")
    return report

GRPO_PATH = run_stage("grpo", CONFIG_PATH, GRPO_DIR,
                      steps=min(GRPO_INTERVAL, GRPO_LIMIT), initialize_from=SFT_PATH)
comparison = evaluate_and_save_grpo()
show_training_curve(GRPO_DIR, "grpo")

### 50회 더 학습 — 이 셀을 반복 실행

현재 저장된 GRPO에서 50회를 더 학습하고 검증 선택 모델을 갱신합니다.
기본 총 250회까지 반복할 수 있습니다. `smoke`에서는 한 번에 1회씩 총 2회 실행합니다.
구간별 도착률·추락률·평균 보상과 선택된 모델을 확인하세요.

In [ ]:
payload = torch.load(GRPO_DIR / "grpo.pt", map_location="cpu", weights_only=True)
completed_steps = payload["step"]
del payload
next_steps = min(completed_steps + GRPO_INTERVAL, GRPO_LIMIT)
if next_steps > completed_steps:
    GRPO_PATH = run_stage("grpo", CONFIG_PATH, GRPO_DIR, steps=next_steps, resume=True)
    comparison = evaluate_and_save_grpo()
    show_training_curve(GRPO_DIR, "grpo")
else:
    print(f"GRPO {completed_steps}회 완료. 아래 실행 화면과 최종 테스트로 이동하세요.")

### 3단계 평가와 나란히 실행

시작 SFT와 검증으로 선택한 모델을 같은 지도와 환경 난수에서 실행합니다.
`selection.json`에 선택된 단계와 업데이트 수가 기록됩니다.
지도와 seed를 고르고 ‘지도 초기화’를 누른 뒤 한 걸음씩 또는 자동으로 실행하세요.

In [ ]:
selected = json.loads(SELECTION_PATH.read_text(encoding="utf-8"))
print("선택 모델:", selected["source"], "업데이트:", selected["step"])
comparison_demo = launch_navigation({
    "시작 SFT": str(GRPO_DIR / "initial_sft.pt"),
    "검증 선택": str(SELECTED_PATH),
}, CONFIG_PATH)

## 4. 최종 테스트

GRPO 구간별 비교와 모델 선택을 마친 뒤 실행하세요. 검증과 다른 지도 seed를 사용합니다.
시작 SFT, 마지막 GRPO, 검증 선택 모델을 BFS·optimal과 비교합니다.
BFS는 현재 위치에서 다시 계획하고 optimal은 미끄러짐·보상·남은 행동 수를 고려합니다.

In [ ]:
final_test = report_navigation({
    "시작 SFT": str(GRPO_DIR / "initial_sft.pt"),
    "마지막 GRPO": str(GRPO_DIR / "grpo.pt"),
    "검증 선택": str(SELECTED_PATH),
}, CONFIG_PATH, split="test")

## 5. 선택적 단계 생략 비교

비교할 실험의 스위치를 `True`로 바꾸고 해당 셀을 실행하세요.
사전학습 생략은 무작위 초기화 → SFT → GRPO,
SFT 생략은 사전학습 → GRPO 순서입니다.
생략 실험은 별도 하위 폴더에 저장합니다. 사전학습 생략의 SFT는 위에서 지정한 총 업데이트 수를 사용합니다. 각 실험의 업데이트 수와 검증 지표를 비교하세요.

In [ ]:
RUN_WITHOUT_PRETRAIN = False
if RUN_WITHOUT_PRETRAIN:
    ablation_dir = OUTPUT_DIR / "without_pretrain"
    seed_runtime(config["seed"])
    random_model, _ = create_model(config, choose_device())
    random_path = ablation_dir / "random_initialization.pt"
    save_checkpoint(random_path, random_model, config, "random_initialization", 0)
    del random_model
    run_stage("sft", CONFIG_PATH, ablation_dir, steps=SFT_TOTAL_STEPS, initialize_from=random_path)
    run_stage("grpo", CONFIG_PATH, ablation_dir)
    report_navigation({"선택 모델": str(SELECTED_PATH), "사전학습 생략": str(ablation_dir / "grpo.pt")}, CONFIG_PATH)

In [ ]:
RUN_WITHOUT_SFT = False
if RUN_WITHOUT_SFT:
    ablation_dir = OUTPUT_DIR / "without_sft"
    run_stage("grpo", CONFIG_PATH, ablation_dir, initialize_from=OUTPUT_DIR / "pretrain.pt")
    report_navigation({"선택 모델": str(SELECTED_PATH), "SFT 생략": str(ablation_dir / "grpo.pt")}, CONFIG_PATH)

## 6. 체크포인트 다운로드

`DOWNLOAD_RESULTS = True`로 바꾸고 실행해 모델·설정·학습 로그·평가 결과를 ZIP으로 내려받으세요.
다음 런타임에서 ZIP을 출력 폴더에 복원한 뒤 평가 셀을 실행하면 저장된 모델을 다시 사용할 수 있습니다.

In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files
    archive_path = shutil.make_archive(str(OUTPUT_DIR.parent / f"maze_results_{PROFILE}"), "zip", OUTPUT_DIR)
    files.download(archive_path)